# COMP 6940: Big Data and Data Visualization

### Project  

#### Airline Delay Analysis and Performance Optimization  
#### Dataset: Carrier On-Time Performance Dataset  

## **Notebook:** 01 — Data Cleaning and Preprocessing  

**Objective:**  

This notebook focuses on preparing the raw dataset for analysis by performing data cleaning, transformation, and         feature engineering to produce a structured, analysis-ready dataset.


---

## 01 — Data Cleaning and Preprocessing

This section outlines the data cleaning and preprocessing steps applied to the Carrier On-Time Performance dataset. The objective of this stage is to transform the raw dataset into a structured and analysis-ready format suitable for delay attribution and further downstream statistical analysis.

Key preprocessing steps include handling missing values, standardizing variable formats, converting date and time fields, and generating new features required for analysis.

The final cleaned dataset is exported as `cleaned_flight_data.csv`, which will be used in subsequent stages of the project.

In [2]:
import numpy as np
import pandas as pd

### Load Raw Data

The raw dataset is imported into the environment for preprocessing. This dataset contains flight-level information including scheduling details, delays, and delay causes.

At this stage, no transformations are applied yet; the purpose is to inspect the structure, data types, and initial quality of the dataset before cleaning.

In [3]:
df = pd.read_csv("airline_2m.csv", encoding="ISO-8859-1", low_memory=False)
df.shape

(2000000, 109)

### Drop Empty Columns and Normalize Delay Cause Variables

Columns that contain entirely missing or irrelevant values are removed to reduce noise and improve computational efficiency.

Additionally, missing values in delay cause variables are standardized to ensure consistency in later calculations. This step is important because delay-related metrics will be aggregated and analyzed, and inconsistent missing values could distort results.

In [4]:
df = df.dropna(how="all", axis=1)

delay_cause_cols = [
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
]
for col in delay_cause_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

if "CancellationCode" in df.columns:
    df["CancellationCode"] = df["CancellationCode"].fillna("unknown")

### Date and Categorical Variable Processing

Date-related variables are converted into appropriate datetime formats to enable time-based analysis. Categorical variables are also standardized to ensure consistent labeling across the dataset.

This step ensures that temporal trends (e.g., daily or monthly patterns) and categorical comparisons can be accurately computed in later stages.

In [5]:
df["FlightDate"] = pd.to_datetime(df["FlightDate"])

for col in (
    "Reporting_Airline",
    "IATA_CODE_Reporting_Airline",
    "Origin",
    "Dest",
    "CancellationCode",
):
    if col in df.columns:
        df[col] = df[col].astype("category")

df

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div1WheelsOff,Div1TailNum,Div2Airport,Div2AirportID,Div2AirportSeqID,Div2WheelsOn,Div2TotalGTime,Div2LongestGTime,Div2WheelsOff,Div2TailNum
0,1998,1,1,2,5,1998-01-02,NW,19386,NW,N297US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009,2,5,28,4,2009-05-28,FL,20437,FL,N946AT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013,2,6,29,6,2013-06-29,MQ,20398,MQ,N665MQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010,3,8,31,2,2010-08-31,DL,19790,DL,N6705Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2006,1,1,15,7,2006-01-15,US,20355,US,N504AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999995,2008,1,3,23,7,2008-03-23,WN,19393,WN,N712SW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999996,1999,1,1,5,2,1999-01-05,CO,19704,CO,N14308,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999997,2003,4,11,14,5,2003-11-14,US,20355,US,N528AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999998,2012,2,5,15,2,2012-05-15,WN,19393,WN,N281WN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### State Code Standardization

State codes are converted to uppercase and stored using a string data type. This approach preserves missing values while ensuring consistency in formatting.

Standardizing state codes is essential for grouping, filtering, and geographic analysis in later stages of the project.

In [6]:
for col in ("OriginState", "DestState"):
    if col in df.columns:
        df[col] = df[col].astype("string").str.upper()

df[["OriginState","DestState"]]

,OriginState,DestState
0,MN,UT
1,WI,FL
2,CO,TX
3,CA,MI
4,NJ,NC
...,...,...
1999995,NV,AZ
1999996,NJ,TX
1999997,SC,NC
1999998,IL,TN


### Scheduled Time Conversion (HHMM to Minutes)

Scheduled departure and arrival times, originally stored in HHMM format, are converted into minutes since midnight. This transformation allows for easier numerical analysis and comparison of time-based variables.

Invalid or missing time values are converted to `NaN` to prevent incorrect calculations.

Arrival and departure delay variables are retained in their original form, including negative values, which represent early arrivals or departures. It was important to preserve these values in order to accurately capture flight performance.

In [7]:
def hhmm_to_minutes(series):
    v = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float, copy=False)
    out = np.full(v.shape, np.nan, dtype=float)
    mask = np.isfinite(v) & (v >= 0)
    vi = np.floor(v[mask]).astype(np.int64, copy=False)
    out[mask] = (vi // 100) * 60 + (vi % 100)
    return pd.Series(out, index=series.index)


time_cols = [
    "CRSDepTime",
    "DepTime",
    "WheelsOff",
    "WheelsOn",
    "CRSArrTime",
    "ArrTime",
]
for col in time_cols:
    if col in df.columns:
        df[col] = hhmm_to_minutes(df[col])

df

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div1WheelsOff,Div1TailNum,Div2Airport,Div2AirportID,Div2AirportSeqID,Div2WheelsOn,Div2TotalGTime,Div2LongestGTime,Div2WheelsOff,Div2TailNum
0,1998,1,1,2,5,1998-01-02,NW,19386,NW,N297US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009,2,5,28,4,2009-05-28,FL,20437,FL,N946AT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013,2,6,29,6,2013-06-29,MQ,20398,MQ,N665MQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010,3,8,31,2,2010-08-31,DL,19790,DL,N6705Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2006,1,1,15,7,2006-01-15,US,20355,US,N504AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999995,2008,1,3,23,7,2008-03-23,WN,19393,WN,N712SW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999996,1999,1,1,5,2,1999-01-05,CO,19704,CO,N14308,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999997,2003,4,11,14,5,2003-11-14,US,20355,US,N528AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999998,2012,2,5,15,2,2012-05-15,WN,19393,WN,N281WN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Calendar Features and Seasonal Classification

Additional calendar-based features are derived from the `FlightDate` and `Month` variables. These include attributes such as day, month, and season.

Seasonal classification enables the analysis of trends across different times of the year, which is particularly relevant for identifying patterns in flight delays.

In [8]:
df["Year"] = df["FlightDate"].dt.year
df["Quarter"] = df["FlightDate"].dt.quarter
df["Month"] = df["FlightDate"].dt.month
df["DayOfWeek"] = df["FlightDate"].dt.dayofweek + 1


def month_to_season(m):
    if m in (12, 1, 2):
        return "Winter"
    if m in (3, 4, 5):
        return "Spring"
    if m in (6, 7, 8):
        return "Summer"
    return "Autumn"


df["Season"] = df["Month"].map(month_to_season).astype("category")
df[["Month","Season"]]

,Month,Season
0,1,Winter
1,5,Spring
2,6,Summer
3,8,Summer
4,1,Winter
...,...,...
1999995,3,Spring
1999996,1,Winter
1999997,11,Autumn
1999998,5,Spring


### Delay Indicators and Derived Features

New variables are created to enhance the analysis of flight delays. These include:

- Binary indicators identifying whether a flight experienced a delay
- Aggregated delay cause totals
- Flags for specific delay conditions
- Route-level identifiers
- Time-based features such as departure and arrival hours

These derived features allow for more detailed analysis of delay patterns, including identifying dominant delay causes and examining trends across routes and time periods.

In [9]:
df["IsArrivalDelayed"] = df["ArrDelay"] > 0
df["IsDepartureDelayed"] = df["DepDelay"] > 0

df["TotalCauseDelay"] = (
    df["CarrierDelay"]
    + df["WeatherDelay"]
    + df["NASDelay"]
    + df["SecurityDelay"]
    + df["LateAircraftDelay"]
)

df["HasCarrierDelay"] = df["CarrierDelay"] > 0
df["HasWeatherDelay"] = df["WeatherDelay"] > 0
df["HasNASDelay"] = df["NASDelay"] > 0
df["HasSecurityDelay"] = df["SecurityDelay"] > 0
df["HasLateAircraftDelay"] = df["LateAircraftDelay"] > 0

df["Route"] = df["Origin"].astype(str) + "-" + df["Dest"].astype(str)

df["DepHour"] = (df["CRSDepTime"] // 60).astype("Int64")
df["ArrHour"] = (df["CRSArrTime"] // 60).astype("Int64")

### Data Validation

Basic validation checks performed to ensure that the preprocessing steps were applied correctly. This includes verifying data types, checking for unexpected missing values, and confirming that derived variables behave as expected.

These checks help ensure data integrity before proceeding to analysis.

In [10]:
df[[
    "FlightDate",
    "Season",
    "IsArrivalDelayed",
    "TotalCauseDelay",
    "Route",
    "DepHour",
    "ArrHour",
]].head()

,FlightDate,Season,IsArrivalDelayed,TotalCauseDelay,Route,DepHour,ArrHour
0,1998-01-02,Winter,True,0.0,MSP-SLC,16,18
1,2009-05-28,Spring,False,0.0,MKE-MCO,12,15
2,2013-06-29,Summer,False,0.0,GJT-DFW,16,19
3,2010-08-31,Summer,False,0.0,LAX-DTW,13,20
4,2006-01-15,Winter,True,32.0,EWR-CLT,18,20


### Save Cleaned Dataset

The fully processed dataset is saved as `cleaned_flight_data.csv`. This file serves as the input for subsequent analysis and modeling stages.

Saving the cleaned dataset ensures reproducibility and allows future steps to be performed without repeating preprocessing operations.

In [11]:
df.to_csv("cleaned_flight_data.csv", index=False)